In [4]:
import pandas as pd
import requests
from datetime import date, timedelta
import os
import io
import json
import zipfile
import tqdm
import pyarrow
import openpyxl

In [6]:
def get_forts_history_day(trade_date, url):

    important_columns = [
        "TRADEDATE",
        "SECID",
        "SHORTNAME",
        "ASSETCODE",
        "VALUE",
        "VOLUME",
        "NUMTRADES",
        "OPENPOSITION",
        "OPENPOSITIONVALUE"
    ]

    all_rows: list[pd.DataFrame] = []
    start = 0

    while True:
        params = {
            "date": trade_date,
            "start": start
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        columns = data["history"]["columns"]
        rows = data["history"]["data"]

        if len(rows) == 0:
            break

        page_df = pd.DataFrame(rows, columns=columns)
        existing_columns = [
            col for col in important_columns
            if col in page_df.columns
        ]

        page_df = page_df[existing_columns]

        all_rows.append(page_df)

        start += len(rows)

    if len(all_rows) == 0:
        return pd.DataFrame(columns=important_columns)

    result = pd.concat(all_rows, ignore_index=True)

    numeric_columns = [
        "VALUE",
        "VOLUME",
        "NUMTRADES",
        "OPENPOSITION",
        "OPENPOSITIONVALUE"
    ]

    result = result[result["VOLUME"] != 0].dropna(subset=["VOLUME"]).copy()

    result["TRADEDATE"] = pd.to_datetime(result["TRADEDATE"])


    for col in numeric_columns:
        result[col] = pd.to_numeric(result[col], errors="coerce")

    result["year"] = result["TRADEDATE"].dt.year
    result["month"] = result["TRADEDATE"].dt.month
    result["day"] = result["TRADEDATE"].dt.day


    return result

In [7]:
url_futures = "https://iss.moex.com/iss/history/engines/futures/markets/forts/securities.json"
df = get_forts_history_day("2024-03-26", url_futures)
print(df.shape)
df.to_csv('futures_data.csv', index=False, encoding='utf-8')

(236, 12)


In [8]:
def date_range(start_date, end_date):
    current_date = start_date

    while current_date <= end_date:
        yield current_date
        current_date += timedelta(days=1)

In [9]:
def collect_forts_history_period(url_futures_new_l, start_date, end_date):
    all_days = []

    for current_date in date_range(start_date, end_date):
        trade_date = current_date.isoformat()

        print(f"Loading {trade_date} futures")
        day_df = get_forts_history_day(trade_date, url_futures_new_l)

        if not day_df.empty:
            all_days.append(day_df)

    if len(all_days) == 0:
        return pd.DataFrame()

    result = pd.concat(all_days, ignore_index=True)

    return result

In [10]:
url_futures_new = "https://iss.moex.com/iss/history/engines/futures/markets/forts/securities.json"
start_day = 1
end_day = 31
start_month = 4
end_month = 4
start_year = 2026
end_year = 2026
df = collect_forts_history_period(
    url_futures_new,
    start_date=date(start_year, start_month, start_day),
    end_date=date(end_year, end_month, end_day)
)
df.to_csv(f'april_futures.csv', index=False, encoding='utf-8')

Loading 2026-03-01...
Loading 2026-03-02...
Loading 2026-03-03...
Loading 2026-03-04...
Loading 2026-03-05...


In [28]:
def get_options_assets(trade_date):
    url_options_series = "https://iss.moex.com/iss/statistics/engines/futures/markets/options/series.json"

    response = requests.get(url_options_series, params={"date": trade_date})
    response.raise_for_status()

    data = response.json()

    df_series = pd.DataFrame(
        data["series"]["data"],
        columns=data["series"]["columns"]
    )

    assets = (
        df_series["asset_code"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    return assets

In [43]:
def get_options_volumes_asset_day(asset, trade_date):
    url = (
        "https://iss.moex.com/iss/statistics/engines/futures/"
        f"markets/options/assets/{asset}/volumes.json"
    )

    response = requests.get(url, params={"date": trade_date})
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(
        data["asset_volumes"]["data"],
        columns=data["asset_volumes"]["columns"]
    )
    df = df[df['voltoday'] != 0].copy()
    df = df[df['voltoday'] != '0'].copy()
    return df

In [44]:
def collect_options_history_period(start_date, end_date):
    all_days = []
    for trade_date in date_range(start_date, end_date):
        assets = get_options_assets(trade_date)
        for asset in assets:
            day_asset_df = get_options_volumes_asset_day(asset, trade_date)
            if not day_asset_df.empty:
                all_days.append(day_asset_df)

            print(f"{asset} parsed")
    result: pd.DataFrame = pd.concat(all_days, ignore_index=True)
    numeric_columns = [
        "valtoday",
        "voltoday",
        "openposition",
        "oichange"
    ]

    for col in numeric_columns:
        result[col] = pd.to_numeric(result[col], errors="coerce")

    result["tradedate"] = pd.to_datetime(result["tradedate"], errors="coerce")
    result["expiration_date"] = pd.to_datetime(result["expiration_date"], errors="coerce")

    result["year"] = result["tradedate"].dt.year
    result["month"] = result["tradedate"].dt.month
    result["day"] = result["tradedate"].dt.day
    if len(all_days) == 0:
        return pd.DataFrame()

    result = pd.concat(all_days, ignore_index=True)

    return result

In [45]:
# парсим серии опционов
options_df = collect_options_history_period(

    start_date=date(2026, 4, 1),
    end_date=date(2026, 4, 4)
)

print(options_df.shape)

options_df.to_csv("april_options.csv", index=False, encoding="utf-8-sig")

AFKS parsed
AFLT parsed
ALRS parsed
ASTR parsed
BR parsed
CHMF parsed
CNY parsed
DAX parsed
DIAS parsed
ED parsed
Eu parsed
FEES parsed
GAZR parsed
GL parsed
GMKN parsed
GOLD parsed
HANG parsed
HYDR parsed
IMX parsed
IRAO parsed
ISKJ parsed
LKOH parsed
MAGN parsed
MGNT parsed
MIX parsed
MOEX parsed
MSNG parsed
MTLR parsed
MTSI parsed
MXI parsed
NASD parsed
NG parsed
NLMK parsed
NOTK parsed
PIKK parsed
PLT parsed
PLZL parsed
POSI parsed
ROSN parsed
RTKM parsed
RTS parsed
RUAL parsed
SBPR parsed
SBRF parsed
SIBN parsed
SILV parsed
SL parsed
SMLT parsed
SNGP parsed
SNGR parsed
SPBE parsed
SPYF parsed
STOX parsed
SVCB parsed
Si parsed
TATN parsed
TATP parsed
TCSI parsed
TRNF parsed
UCNY parsed
VKCO parsed
VTBR parsed
WHEAT parsed
YDEX parsed
AFKS parsed
AFLT parsed
ALRS parsed
ASTR parsed
BR parsed
CHMF parsed
CNY parsed
DAX parsed
DIAS parsed
ED parsed
Eu parsed
FEES parsed
GAZR parsed
GL parsed
GMKN parsed
GOLD parsed
HANG parsed
HYDR parsed
IMX parsed
IRAO parsed
ISKJ parsed
LKOH parsed

In [26]:
url_options_series = "https://iss.moex.com/iss/statistics/engines/futures/markets/options/series.json"
response = requests.get(url_options_series)
response.raise_for_status()
data_series = response.json()
print(data_series.keys())
print(data_series['series']['columns'])
df_series = pd.DataFrame( data_series["series"]["data"], columns=data_series["series"]["columns"] )
series_keys = df_series[ [ "asset_code", "series_type", "expiration_date", "margin_style", "exec_type", "option_on_spot", "settle_type", "underlying_asset" ] ].drop_duplicates()
# парсим объемы по assed_code
asset = df_series["asset_code"].iloc[0]
url_options_volumes = f"https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/{asset}/volumes.json"
trade_date = "2023-09-15"
params = { "date": trade_date }
response = requests.get(url_options_volumes, params=params)
response.raise_for_status()
data_volumes = response.json()
print(data_volumes.keys())
print(data_volumes['asset_volumes']['columns'])
df_volumes = pd.DataFrame( data_volumes["asset_volumes"]["data"], columns=data_volumes["asset_volumes"]["columns"] )
df_volumes.head()

dict_keys(['series'])
['name', 'start_date', 'expiration_date', 'series_type', 'exec_type', 'margin_style', 'asset_code', 'underlying_asset', 'is_traded', 'central_strike', 'strike_from', 'strike_till', 'option_on_spot', 'settle_type']
dict_keys(['asset_volumes'])
['tradedate', 'asset', 'series_type', 'expiration_date', 'valtoday', 'voltoday', 'openposition', 'oichange', 'updatetime']


,tradedate,asset,series_type,expiration_date,valtoday,voltoday,openposition,oichange,updatetime
0,2023-09-15,ALRS,W,2023-09-27,3436.0,43,1076,86,2023-09-15 18:50:00
1,2023-09-15,ALRS,W,2023-10-04,0.0,0,200,0,2023-09-15 18:49:00
2,2023-09-15,ALRS,M,2023-09-20,66398.0,882,37854,582,2023-09-15 18:50:00
3,2023-09-15,ALRS,M,2023-10-18,7800.0,100,11132,200,2023-09-15 18:50:00
4,2023-09-15,ALRS,M,2023-11-15,0.0,0,0,0,2023-09-15 18:48:00
